In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import scipy.sparse as sp
from anndata import AnnData

# Set publication-grade plotting parameters
scv.settings.set_figure_params('scvelo')

# ==============================================================================
# 1. PARAMETERS & PATH SPECIFICATION
# ==============================================================================
input_path = "/mnt/d........adata_final_aligned_velocity.h5ad"
save_preprocessed = "/mnt/d/...........adata_preprocessed_moments.h5ad"
save_final = "/mnt/d/..............adata_HNRNP_velocity_final.h5ad"

print("--- [Initialization] Environment & Data Ingestion ---")
print(f"NumPy Version: {np.__version__}")

# Load aligned AnnData object
adata = sc.read_h5ad(input_path)
print(f"Loaded Matrix Dimensions: Cells={adata.n_obs} | Genes={adata.n_vars}")
print(f"Available Layers: {list(adata.layers.keys())}")

# Display raw splice structure ratios
scv.pl.proportions(adata, show=False)

# Keep an isolated copy of the un-sliced full feature space for target recovery
adata_full = adata.copy()

# ==============================================================================
# 2. FILTERING & HIGHLY VARIABLE GENE SELECTION
# ==============================================================================
print("\n--- [Preprocessing] Filtering and Normalization ---")
scv.pp.filter_and_normalize(adata, min_shared_counts=20)

sc.pp.highly_variable_genes(adata, n_top_genes=3000, flavor='seurat_v3')
num_hvg = adata.var['highly_variable'].sum()
print(f"Identified Highly Variable Genes (HVGs): {num_hvg}")

# Slice matrix down to selected high-variance features
adata = adata[:, adata.var['highly_variable']].copy()

# ==============================================================================
# 3. TARGET FEATURE RECOVERY PROGRAM (HNRNP FAMILY)
# ==============================================================================
print("\n--- [Target Recovery] Checking Vital Kinetic Markers ---")
target_genes = ['HNRNPD', 'HNRNPU', 'HNRNPA2B1', 'HNRNPK']
missing_genes = [g for g in target_genes if g not in adata.var_names]

if not missing_genes:
    print("All required HNRNP target features are natively present in the HVG slice.")
else:
    print(f"Target features missing from HVG space: {missing_genes}. Initiating splice reconstruction...")
    
    # Isolate missing targets from the original complete feature block using exact cell matching
    hnrnp_subset = adata_full[adata.obs_names, missing_genes].copy()
    
    # Physically concatenate sparse matrices across the feature axis
    new_X = sp.hstack([sp.csr_matrix(adata.X), sp.csr_matrix(hnrnp_subset.X)])
    
    new_layers = {}
    for layer in ['spliced', 'unspliced']:
        if layer in adata.layers:
            new_layers[layer] = sp.hstack([
                sp.csr_matrix(adata.layers[layer]), 
                sp.csr_matrix(hnrnp_subset.layers[layer])
            ])
            
    # Concatenate feature-level metadata dataframes
    new_var = pd.concat([adata.var, hnrnp_subset.var])
    
    # Reconstruct optimized unified AnnData object container
    adata_final = AnnData(
        X=new_X,
        obs=adata.obs.copy(),
        var=new_var,
        layers=new_layers,
        obsm=adata.obsm.copy(),
        uns=adata.uns.copy()
    )
    
    adata = adata_final
    print(f"Matrix dim reconstruction successful. Updated dimensions: {adata.shape}")

# ==============================================================================
# 4. DIMENSIONALITY REDUCTION & MOMENT ESTIMATION
# ==============================================================================
print("\n--- [Graph Computations] Generating Local Moments ---")
sc.tl.pca(adata)

# Refresh neighbor graph matrices globally to safely integrate recovered features
sc.pp.neighbors(adata, n_pcs=30, n_neighbors=30)
scv.pp.moments(adata, n_pcs=None, n_neighbors=None)

# Cache intermediate post-moments checkpoint matrix to local disk
os.makedirs(os.path.dirname(save_preprocessed), exist_ok=True)
adata.write_h5ad(save_preprocessed)
print(f"Intermediate moments matrix archived safely at: {save_preprocessed}")

# ==============================================================================
# 5. VELOCITY KINETICS MODELING
# ==============================================================================
print("\n--- [Kinetics Modeling] Running Stochastic Velocity Engine ---")
# Resolve transcriptional velocity vectors using a stochastic modeling method
scv.tl.velocity(adata, mode='stochastic')

# Compute the correlation matrix projection velocity graph
scv.tl.velocity_graph(adata)

# ==============================================================================
# 6. STRUCTURAL DATA REPAIR & SERIALIZATION
# ==============================================================================
print("\n--- [Data Integrity] Formatting Metadata Schemas ---")

# Type-cast boolean indicator columns to clean HDF5 array serializing conflicts
if 'highly_variable' in adata.var.columns:
    adata.var['highly_variable'] = adata.var['highly_variable'].fillna(False).astype(bool)

# Clean and homogenize generic unstructured object datatypes to plain string formats
for col in adata.var.columns:
    if adata.var[col].dtype == 'object':
        adata.var[col] = adata.var[col].fillna('').astype(str)

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].fillna('').astype(str)

# Save the finalized model object to persistent local disk storage
os.makedirs(os.path.dirname(save_final), exist_ok=True)
adata.write_h5ad(save_final)

print(f"[SUCCESS] Finalized RNA Velocity pipeline execution. Output locked at: {save_final}")
print(adata.obs.head())